# 🔬 Eksperimen 1: Repeated LOPO-CV & Uji Konsistensi (Hingga 10x Perulangan)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Tujuan Eksperimen**: Menguji kestabilan, variansi stokastik, dan konsistensi empiris dari 48 pipeline faktorial (C1/C2 × S1/S2/S3 × B1/B2/B3/B4 × M1/M2) lintas **10 independent random seeds**.  
**Daftar Seeds Baku (10x)**: `[42, 100, 2024, 777, 999, 123, 456, 789, 2025, 2026]`  
**Protokol Validasi**: Leave-One-Project-Out Cross-Validation (Zero Data Leakage).

## 1. Setup Lingkungan & Import Modul Eksperimen

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Hubungkan ke modul src
src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

from data_loader import load_dataset
from pipeline_builder import get_pipeline_specs
from repeated_lopo_runner import run_repeated_lopo_experiments, DEFAULT_SEEDS_10
from consistency_analyzer import analyze_pipeline_consistency

print(f"[OK] 10 Baku Random Seeds: {DEFAULT_SEEDS_10}")
all_specs = get_pipeline_specs()
print(f"[OK] Total Pipeline Specs Terdefinisi: {len(all_specs)} Pipelines")

## 2. Verifikasi Data C1 (Row-Level Drop) & C2 (Project-Level Drop)

In [ ]:
df_c1, feat_c1, proj_c1 = load_dataset('C1')
df_c2, feat_c2, proj_c2 = load_dataset('C2')

print(f"[C1 Row-Level Drop]     : {len(df_c1):,} baris data | {len(proj_c1)} proyek | Flaky: {df_c1['flaky'].sum()}")
print(f"[C2 Project-Level Drop] : {len(df_c2):,} baris data | {len(proj_c2)} proyek | Flaky: {df_c2['flaky'].sum()}")

## 3. Eksekusi Uji Konsistensi (Repeated LOPO-CV)
Anda dapat memilih mode eksekusi di bawah ini:
- **Mode Cepat (Smoke Test)**: Memvalidasi integritas kode pada 2 fold & 2 seeds.
- **Mode Top Pipelines (Rekomendasi)**: Menguji 10 pipeline terbaik lintas 10 random seeds.
- **Mode Full Factorial**: Menguji seluruh 48 pipeline lintas 10 seeds.

In [ ]:
# Pilihan A: Mode Cepat / Smoke Test (2 Pipelines, 2 Seeds)
# Untuk verifikasi cepat sebelum menjalankan full 10x
results_quick = run_repeated_lopo_experiments(
    pipelines=[all_specs[0], all_specs[12]],
    seeds=DEFAULT_SEEDS_10[:2],
    output_dir="../results_quick_test",
    use_gpu=True,
    quick_test=True
)
display(results_quick["all_runs"].head())

In [ ]:
# Pilihan B: Eksekusi Top 10 Pipelines Eksperimen 1 Lintas 10 Random Seeds
top_pipeline_ids = [
    "P37_C2_S2_B3_M1", "P38_C2_S2_B3_M2", "P46_C2_S3_B3_M2", "P30_C2_S1_B3_M2",
    "P45_C2_S3_B3_M1", "P29_C2_S1_B3_M1", "P32_C2_S1_B4_M2", "P40_C2_S2_B4_M2",
    "P48_C2_S3_B4_M2", "P13_C1_S2_B3_M1"
]
selected_pipelines = [p for p in all_specs if p["pipeline_id"] in top_pipeline_ids]

print(f"Memulai repeated evaluation untuk {len(selected_pipelines)} top pipelines lintas 10 seeds...")
results_10x = run_repeated_lopo_experiments(
    pipelines=selected_pipelines,
    seeds=DEFAULT_SEEDS_10, # 10 Random Seeds
    output_dir="../results",
    use_gpu=True
)

In [ ]:
# Pilihan C: Eksekusi Penuh Seluruh 48 Pipelines Eksperimen 1 Lintas 10 Random Seeds (10.320 Model Folds)
# PERHATIAN:
# - Beban komputasi: 10 Repetisi x [24 Folds C1 + 19 Folds C2] x 24 Pipelines = 10.320 Model Folds.
# - Estimasi waktu: ~2.5 jam pada GPU RTX 4050.
# - Ubah run_full_48 = True jika ingin mengeksekusinya langsung dari notebook:

run_full_48 = False

if run_full_48:
    print(f"🚀 Memulai repeated LOPO-CV penuh untuk seluruh {len(all_specs)} pipelines pada 10 seeds...")
    results_full = run_repeated_lopo_experiments(
        pipelines=all_specs,
        seeds=DEFAULT_SEEDS_10,
        output_dir="../results",
        use_gpu=True
    )
    display(results_full["all_runs"].head(10))
else:
    print("[INFO] Pilihan C diset False secara default untuk mencegah eksekusi panjang yang tidak disengaja.")
    print("       Lanjutkan ke Sel Nomor 4 di bawah untuk menganalisis tabel konsistensi dan stabilitas.")


## 4. Analisis Konsistensi & Tabel Stabilitas Empiris
Menghitung Mean (\mu), Standar Deviasi (\sigma), Coefficient of Variation ($CV = \frac{\sigma}{\mu} \times 100\%$), serta rentang nilai (Min-Max) dan 95% Confidence Interval.

In [ ]:
df_cons = analyze_pipeline_consistency(
    all_runs_csv="../results/all_seed_pipeline_runs.csv",
    output_dir="../results"
)

cols_show = [
    "Pipeline_ID", "Cleaning", "Scaling", "Imbalance", "Model", "Num_Seeds",
    "Mean_F1_Mean", "Mean_F1_Std", "Mean_F1_CV_%", "Mean_F1_CI95",
    "Mean_Recall_Mean", "Mean_Precision_Mean", "Mean_PR_AUC_Mean"
]
display(df_cons[[c for c in cols_show if c in df_cons.columns]].head(10))

## 5. Visualisasi Kestabilan Lintas 10 Repetisi

In [ ]:
df_runs = pd.read_csv("../results/all_seed_pipeline_runs.csv")

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_runs, x="Pipeline_ID", y="Mean_F1", hue="Model")
plt.xticks(rotation=45, ha="right")
plt.title("Distribusi F1-Score Lintas 10 Repetisi Random Seeds (Uji Konsistensi)", fontsize=14, fontweight="bold")
plt.ylabel("Mean F1-Score")
plt.xlabel("Pipeline ID")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()